# Fine Tuning Gemma 3

In [1]:
import os

is_colab = False
is_sagemaker = False
device = "mps"

env_keys = os.environ.keys()

if any(k.startswith("COLAB_") for k in env_keys):
    is_colab = True
    device = "cuda"
    print("Running in Google Colab")

elif "SM_CURRENT_HOST" in os.environ:
    is_sagemaker = True
    device = "cuda"
    print("Running in SageMaker")

else:
    print("Running locally")


Running in Google Colab


In [2]:
device

'cuda'

In [3]:
if is_colab or is_sagemaker:
    !nvidia-smi

Sat Dec 20 07:26:26 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
%%capture
if is_colab:
    !pip install -U bitsandbytes transformers==4.57.3 datasets peft acecelerate
    # !pip install -U transformers==4.57.3
    !pip install -U bitsandbytes

In [ ]:
if is_colab:
    from google.colab import userdata
    # os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    from transformers.utils.quantization_config import BitsAndBytesConfig
    quantization_config = BitsAndBytesConfig(load_in_8bit=True)
os.environ["HF_TOKEN"] = ""

In [5]:
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
from peft import PeftModel, PeftConfig
from transformers.trainer import Trainer
from transformers.training_args import TrainingArguments
from transformers.data.data_collator import DataCollatorForLanguageModeling
from peft import prepare_model_for_kbit_training
from tqdm.auto import tqdm
from transformers.optimization import get_cosine_schedule_with_warmup
from datasets import load_from_disk

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

## Loading the Model

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer
base_model_id = "google/gemma-3-270m-it"

tokenizer = AutoTokenizer.from_pretrained(
    base_model_id
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    attn_implementation="eager",
    quantization_config=quantization_config if is_colab else None,
)

tokenizer.padding_side = "right"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

In [9]:
for name, param in model.named_parameters():
    print(name, param.device)
    break

model.embed_tokens.weight cuda:0


In [10]:
print(tokenizer.get_chat_template())

{{ bos_token }}
{%- if messages[0]['role'] == 'system' -%}
    {%- if messages[0]['content'] is string -%}
        {%- set first_user_prefix = messages[0]['content'] + '

' -%}
    {%- else -%}
        {%- set first_user_prefix = messages[0]['content'][0]['text'] + '

' -%}
    {%- endif -%}
    {%- set loop_messages = messages[1:] -%}
{%- else -%}
    {%- set first_user_prefix = "" -%}
    {%- set loop_messages = messages -%}
{%- endif -%}
{%- for message in loop_messages -%}
    {%- if (message['role'] == 'user') != (loop.index0 % 2 == 0) -%}
        {{ raise_exception("Conversation roles must alternate user/assistant/user/assistant/...") }}
    {%- endif -%}
    {%- if (message['role'] == 'assistant') -%}
        {%- set role = "model" -%}
    {%- else -%}
        {%- set role = message['role'] -%}
    {%- endif -%}
    {{ '<start_of_turn>' + role + '
' + (first_user_prefix if loop.first else "") }}
    {%- if message['content'] is string -%}
        {{ message['content'] | trim }}


**Testing the prompt before fine-tuning:**

In [11]:
messages = [
    {"role": "system", "content": "You are a assistant responsible for classifying mental health status."},
    {"role": "user", "content": "I am depressed and want to die"}
]


input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(device)
attention_mask = torch.ones_like(input_ids).to(device)

outputs = model.generate(
    input_ids,
    attention_mask=attention_mask,
    max_new_tokens=100,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id # Using eos_token_id as pad_token_id for Gemma
)

input_len = input_ids.shape[1]
generated_tokens_tensor = outputs[0, input_len:]
decoded_response = tokenizer.decode(generated_tokens_tensor, skip_special_tokens=True)

print(decoded_response)

+II.1 - Problem-settling activities like talking to a near face-time-in-themologist' pocketspersonsurroworkswear-in-apprx.gov/Seejustridealeadersaintghernoldshellholegayashes-|\_ Februarinerfolgerjesonsagedientiemunisfsourчнымитеерgieseigungenenforfallsucheinlandsングartenanntकप्तान suficiente desteroidone踪pirelaceetimequань direito荫谥reconstructionary


In [12]:
print(tokenizer.apply_chat_template(messages,tokenize = False, add_generation_prompt=True))

<bos><start_of_turn>user
You are a assistant responsible for classifying mental health status.

I am depressed and want to die<end_of_turn>
<start_of_turn>model



## Data Setup

In [12]:
ds = load_dataset("nbertagnolli/counsel-chat")

# drop null
ds = ds.filter(lambda x: x['questionText'] is not None)
ds = ds.filter(lambda x: x['topic'] is not None)

README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


20220401_counsel_chat.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2775 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2775 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2636 [00:00<?, ? examples/s]

In [8]:
ds

DatasetDict({
    train: Dataset({
        features: ['questionID', 'questionTitle', 'questionText', 'questionLink', 'topic', 'therapistInfo', 'therapistURL', 'answerText', 'upvotes', 'views'],
        num_rows: 2636
    })
})

In [9]:
ds["train"][0]

{'questionID': 0,
 'questionTitle': 'Do I have too many issues for counseling?',
 'questionText': 'I have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac.    I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years.\n   I’ve never had counseling about any of this. Do I have too many issues to address in counseling?',
 'questionLink': 'https://counselchat.com/questions/do-i-have-too-many-issues-for-counseling',
 'topic': 'depression',
 'therapistInfo': 'Jennifer MolinariHypnotherapist & Licensed Counselor',
 'therapistURL': 'https://counselchat.com/therapists/jennifer-molinari',
 'answerText': 'It is very common for\xa0people to have multiple issues that they want to (and need to) address in counseling.\xa0 I have had clients ask that same question and through more exploration, there is often an underlying fear that they\xa0 "can\

In [13]:
SYSTEM_PROMPT = "You are an assistant responsible for classifying mental health status."

def build_chat(batch):
    questions = batch["questionText"]
    topics = batch["topic"]

    batch_messages = []
    batch_responses = []
    batch_conversations = []

    for q, t in zip(questions, topics):
        batch_messages.append([
            {"role": "user", "content": SYSTEM_PROMPT + "\n" + q},
        ])

        batch_responses.append(f"Based on what you've described, this sounds like '{t}'.")

        batch_conversations.append([
            {"role": "user", "content": SYSTEM_PROMPT + "\n" + q},
            {"role": "assistant", "content": f"Based on what you've described, this sounds like '{t}'."}
        ])

    return {
        "messages": batch_messages,
        "response": batch_responses,
        "conversation": batch_conversations
    }

chat_dataset = ds.map(build_chat, batched=True)

Map:   0%|          | 0/2636 [00:00<?, ? examples/s]

In [14]:
chat_dataset

DatasetDict({
    train: Dataset({
        features: ['questionID', 'questionTitle', 'questionText', 'questionLink', 'topic', 'therapistInfo', 'therapistURL', 'answerText', 'upvotes', 'views', 'messages', 'response', 'conversation'],
        num_rows: 2636
    })
})

In [15]:
MAX_LENGTH = 512
IGNORE_INDEX = -100


def format_conversations(batch):
    """
    Applies the model's chat template to a list of conversational turns.
    """
    texts = []
    for conversation in batch['conversation']:
        formatted_text = tokenizer.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=False 
        ).removeprefix('<bos>')
        texts.append(formatted_text)
    return {"text": texts}

formatted_dataset = chat_dataset.map(format_conversations, batched=True)

Map:   0%|          | 0/2636 [00:00<?, ? examples/s]

In [16]:
formatted_dataset

DatasetDict({
    train: Dataset({
        features: ['questionID', 'questionTitle', 'questionText', 'questionLink', 'topic', 'therapistInfo', 'therapistURL', 'answerText', 'upvotes', 'views', 'messages', 'response', 'conversation', 'text'],
        num_rows: 2636
    })
})

In [17]:
formatted_dataset.column_names

{'train': ['questionID',
  'questionTitle',
  'questionText',
  'questionLink',
  'topic',
  'therapistInfo',
  'therapistURL',
  'answerText',
  'upvotes',
  'views',
  'messages',
  'response',
  'conversation',
  'text']}

In [18]:
print(formatted_dataset["train"][0]["text"])

<start_of_turn>user
You are an assistant responsible for classifying mental health status.
I have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac.    I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years.
   I’ve never had counseling about any of this. Do I have too many issues to address in counseling?<end_of_turn>
<start_of_turn>model
Based on what you've described, this sounds like 'depression'.<end_of_turn>



In [19]:
formatted_dataset["train"][0]["conversation"]

[{'content': 'You are an assistant responsible for classifying mental health status.\nI have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac.    I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years.\n   I’ve never had counseling about any of this. Do I have too many issues to address in counseling?',
  'role': 'user'},
 {'content': "Based on what you've described, this sounds like 'depression'.",
  'role': 'assistant'}]

In [20]:
def tokenize_and_mask_labels(batch):
    """
    Tokenizes the text and creates labels, masking the instruction/user tokens.
    """
    tokenized_results = []

    for conversation in batch['conversation']:
        full_text = tokenizer.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=False
        ).removeprefix('<bos>')


        prompt_conversation = conversation[:-1]

        prompt_text = tokenizer.apply_chat_template(
            prompt_conversation,
            tokenize=False,
            add_generation_prompt=True # Ensures the model's response start token is included
        ).removeprefix('<bos>')

        full_tokenized = tokenizer(
            full_text,
            max_length=MAX_LENGTH,
            truncation=True,
            padding=False,
            return_tensors=None,
        )

        prompt_tokenized = tokenizer(
            prompt_text,
            max_length=MAX_LENGTH,
            truncation=True,
            padding=False,
            return_tensors=None,
        )

        labels = full_tokenized["input_ids"].copy()

        prompt_length = len(prompt_tokenized["input_ids"])

        mask_end = min(prompt_length, len(labels))
        labels[:mask_end] = [IGNORE_INDEX] * mask_end

        input_ids = full_tokenized["input_ids"][:-1]
        labels = labels[1:]
        attention_mask = full_tokenized["attention_mask"][:-1]

        tokenized_results.append({
            "input_ids": input_ids,
            "labels": labels,
            "attention_mask": attention_mask,
            "prompt_input_ids": prompt_tokenized["input_ids"],
            "prompt_attention_mask": prompt_tokenized["attention_mask"]
        })

    return {
        "input_ids": [r["input_ids"] for r in tokenized_results],
        "labels": [r["labels"] for r in tokenized_results],
        "attention_mask": [r["attention_mask"] for r in tokenized_results],
        "prompt_input_ids": [r["prompt_input_ids"] for r in tokenized_results],
        "prompt_attention_mask": [r["prompt_attention_mask"] for r in tokenized_results]
    }



In [21]:
tokenizer.pad_token, tokenizer.eos_token

('<pad>', '<eos>')

In [22]:
tokenized_dataset = formatted_dataset.map(
    tokenize_and_mask_labels,
    batched=True,
    remove_columns=formatted_dataset["train"].column_names
)

Map:   0%|          | 0/2636 [00:00<?, ? examples/s]

In [23]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels', 'attention_mask', 'prompt_input_ids', 'prompt_attention_mask'],
        num_rows: 2636
    })
})

In [24]:
sample = tokenized_dataset['train'][3]


print("\n--- Processed Sample (Input IDs and Labels) ---")
print("Input IDs (first 20):", sample["input_ids"][:20])
print("Labels (first 20):  ", sample["labels"][:20])

print("Input IDs (first 20):", sample["input_ids"])
print("Labels (first 20):  ", sample["labels"])

decoded_input = tokenizer.decode(sample["input_ids"])
print("\nDecoded Input (Full Context):")
print(decoded_input)

mask_start = next(i for i, label in enumerate(sample['labels']) if label != IGNORE_INDEX)
print(f"\nMasking Check:")
print(f"Index of first non-{IGNORE_INDEX} label (start of response): {mask_start}")

first_response_token = sample["labels"][mask_start]
decoded_response_start = tokenizer.decode(first_response_token)

print(f"Decoded token at response start: '{decoded_response_start}'")

is_prompt_masked = all(label == IGNORE_INDEX for label in sample["labels"][:mask_start])
print(f"Are all prompt labels masked (-100)? {is_prompt_masked}")


print("\n--- Dataset ready for Hugging Face Trainer ---")


--- Processed Sample (Input IDs and Labels) ---
Input IDs (first 20): [2, 105, 2364, 107, 3048, 659, 614, 16326, 7757, 573, 99896, 9069, 2404, 4981, 236761, 107, 236777, 735, 834, 1551]
Labels (first 20):   [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]
Input IDs (first 20): [2, 105, 2364, 107, 3048, 659, 614, 16326, 7757, 573, 99896, 9069, 2404, 4981, 236761, 107, 236777, 735, 834, 1551, 4342, 531, 3421, 236761, 564, 735, 496, 4083, 529, 11953, 16407, 236764, 564, 236858, 236757, 496, 16489, 7923, 72399, 532, 564, 1006, 496, 19418, 1728, 542, 218518, 236761, 140, 236777, 735, 496, 1440, 4083, 529, 17998, 532, 564, 236858, 236757, 6534, 531, 735, 17660, 236761, 564, 735, 2708, 1265, 78114, 840, 564, 236858, 560, 1010, 38968, 11578, 573, 4180, 236743, 236800, 236810, 1518, 236761, 107, 139, 236777, 236858, 560, 2752, 1053, 45899, 1003, 1027, 529, 672, 236761, 3574, 564, 735, 2311, 1551, 4342, 531, 3421, 528, 4589

In [25]:
tokenizer.padding_side, tokenizer.truncation_side

('right', 'right')

In [26]:
tokenized_dataset.save_to_disk("../data/processed/tokenized_counsel_chat_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/2636 [00:00<?, ? examples/s]

In [17]:
tokenized_dataset = load_from_disk("../data/processed/tokenized_counsel_chat_dataset")
# tokenized_dataset.load_from_disk("../data/processed/tokenized_counsel_chat_dataset")

In [27]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels', 'attention_mask', 'prompt_input_ids', 'prompt_attention_mask'],
        num_rows: 2636
    })
})

In [28]:
def validate_causal_shift(dataset, tokenizer=None, num_samples=5):
    """
    Checks whether labels are correctly shifted for causal LM.
    """
    import random
    import torch

    indices = random.sample(range(len(dataset)), num_samples)

    for idx in indices:
        sample = dataset[idx]
        input_ids = torch.tensor(sample["input_ids"])
        labels = torch.tensor(sample["labels"])

        valid = labels != -100

        shifted_input = input_ids[1:][valid[:-1]]
        shifted_labels = labels[:-1][valid[:-1]]

        is_shifted = torch.equal(shifted_input, shifted_labels)

        print(f"\nSample {idx}: shifted = {is_shifted}")

        if tokenizer:
            print("Input :", tokenizer.decode(input_ids))
            print("Labels:", tokenizer.decode(labels[labels != -100]))

        if not is_shifted:
            print("Mismatch detected")
        else:
            print("Correctly shifted")


In [29]:
from torch.nn.utils.rnn import pad_sequence

def causal_lm_collator(batch):
    input_ids = [x["input_ids"] for x in batch]
    labels = [x["labels"] for x in batch]
    # prompt_input_ids = [x["prompt_input_ids"] for x in batch]
    # prompt_attention_mask = [x["prompt_attention_mask"] for x in batch]

    input_ids = pad_sequence(
        input_ids, batch_first=True, padding_value=tokenizer.pad_token_id
    )

    labels = pad_sequence(
        labels, batch_first=True, padding_value=-100
    )

    attention_mask = (input_ids != tokenizer.pad_token_id).long()

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask,
        # "prompt_input_ids": prompt_input_ids,
        # "prompt_attention_mask": prompt_attention_mask,
    }

from torch.utils.data import Dataset, DataLoader

class LMDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "labels": torch.tensor(item["labels"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "prompt_input_ids": torch.tensor(item["prompt_input_ids"], dtype=torch.long),
            "prompt_attention_mask": torch.tensor(item["prompt_attention_mask"], dtype=torch.long),
        }

In [30]:
split_dataset = tokenized_dataset["train"].train_test_split(test_size=0.2, seed=42)
split_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels', 'attention_mask', 'prompt_input_ids', 'prompt_attention_mask'],
        num_rows: 2108
    })
    test: Dataset({
        features: ['input_ids', 'labels', 'attention_mask', 'prompt_input_ids', 'prompt_attention_mask'],
        num_rows: 528
    })
})

In [31]:
validate_causal_shift(split_dataset["train"], tokenizer)


Sample 1385: shifted = True
Input : <bos><start_of_turn>user
You are an assistant responsible for classifying mental health status.
It was over 20 years ago, but the pain has resurfaced again now because I have started seeing her Facebook posts about how great her life is. I feel so angry. How can I handle this?<end_of_turn>
<start_of_turn>model
Based on what you've described, this sounds like 'family-conflict'.<end_of_turn>
Labels: Based on what you've described, this sounds like 'family-conflict'.<end_of_turn>

Correctly shifted

Sample 1499: shifted = True
Input : <bos><start_of_turn>user
You are an assistant responsible for classifying mental health status.
My motivation has gone away. It's hard to get out of bed. I really don't know what to do anymore. I'm miserable. My anxiety and depression have taken over my life.<end_of_turn>
<start_of_turn>model
Based on what you've described, this sounds like 'depression'.<end_of_turn>
Labels: Based on what you've described, this sounds lik

In [32]:
batch_size=2

train_dataset = LMDataset(split_dataset["train"])

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=causal_lm_collator,
)

val_dataset = LMDataset(split_dataset["test"])
val_dataloader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    # collate_fn=causal_lm_collator,
)

In [33]:
for batch in train_dataloader:
    print(batch)
    break

{'input_ids': tensor([[     2,    105,   2364,    107,   3048,    659,    614,  16326,   7757,
            573,  99896,   9069,   2404,   4981, 236761,    107, 236777,   2597,
           1133,    564,   1006,    711,    657,    496,   1535,   1883,    529,
           3666, 236761,    564, 236789, 236757,   1401, 172421,    528,   1041,
          12556, 236761,    564, 236789, 236757,    711,   5293,    607,   7564,
            653,    506,   9886,    564,   1386, 236764,    837,   3590,    786,
            711,   5293,    607,   6533,   1663, 236761,    564,   2597,   1133,
            496,   8800,   1346,    529,   1041,   2668, 236761,    564,   1537,
         236789, 236745,   2597,   1133,    564, 236789, 236757,   1535,    657,
           4658,  18602, 236761,    564,   2597,   1133,   2344,    529,    496,
           1589, 236761,    106,    107,    105,   4368,    107,  22515,    580,
           1144,    611, 236789,    560,   4970, 236764,    672,  12054,   1133,
            75

In [34]:
print({k: v.shape for k, v in batch.items()})

{'input_ids': torch.Size([2, 145]), 'labels': torch.Size([2, 145]), 'attention_mask': torch.Size([2, 145])}


## PEFT Setup

In [35]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

**Orginal Model**

In [36]:
model

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear8bitLt(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear8bitLt(in_features=640, out_features=256, bias=False)
          (v_proj): Linear8bitLt(in_features=640, out_features=256, bias=False)
          (o_proj): Linear8bitLt(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear8bitLt(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear8bitLt(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear8bitLt(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_la

In [37]:
# try:
#     peft_model = peft_model.unload()
# except Exception as e:
#     print("Error in get_peft_model:", e)
#     raise e

# peft_model = peft_model.unload()

In [38]:
# peft_model.unload()
train_model = prepare_model_for_kbit_training(model)
peft_model = get_peft_model(train_model, lora_config)

peft_model.enable_input_require_grads()
peft_model.gradient_checkpointing_enable()
peft_model.config.use_cache = False

peft_model.print_trainable_parameters();


trainable params: 737,280 || all params: 268,835,456 || trainable%: 0.2742


**Peft Model**

In [39]:
peft_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma3ForCausalLM(
      (model): Gemma3TextModel(
        (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
        (layers): ModuleList(
          (0-17): 18 x Gemma3DecoderLayer(
            (self_attn): Gemma3Attention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=640, out_features=1024, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=640, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=1024, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
 

## Training Loop

In [40]:
device

'cuda'

In [41]:
total_steps = len(train_dataloader) * 3
total_steps

3162

In [42]:
from torch.amp.grad_scaler import GradScaler
from torch.amp.autocast_mode import autocast


criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.AdamW(
    peft_model.parameters(),
    lr=5e-4,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0.01
)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=50, # ~5% of total steps
    num_training_steps=total_steps
)

scaler = GradScaler()

In [43]:
# scheduler_steps = []
# for step in range(total_steps):
#     scheduler_steps.append(scheduler.get_last_lr()[0])
#     scheduler.step()

In [44]:
# import matplotlib.pyplot as plt
# plt.plot(scheduler_steps)
# plt.xlabel("Step")
# plt.ylabel("Learning Rate")
# plt.title("Learning Rate Schedule")
# plt.show()

In [45]:
num_epochs = 3
accum_steps = 4
num_training_steps = num_epochs * len(train_dataloader) // accum_steps

progress_bar = tqdm(range(num_training_steps))

peft_model.train()
optimizer.zero_grad()

for epoch in range(num_epochs):
    for step, batch in enumerate(train_dataloader):
        with autocast(device_type=device, dtype=torch.float16):
            outputs = peft_model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
            )
            logits = outputs.logits
            labels = batch["labels"].to(device)
            loss = criterion(
                logits.view(-1, logits.size(-1)),
                labels.view(-1)
            )
            loss = loss / accum_steps
        scaler.scale(loss).backward()

        if (step + 1) % accum_steps == 0:
            print(f"Epoch {epoch+1}, Step {step+1}, Loss: {loss.item() * accum_steps:.4f}")

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            progress_bar.update(1)

    if (step + 1) % accum_steps != 0:
        print(f"Epoch {epoch+1}, Step {step+1}, Loss: {loss.item() * accum_steps:.4f}")

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        scheduler.step()
        progress_bar.update(1)


  0%|          | 0/790 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Epoch 1, Step 4, Loss: 4.7947
Epoch 1, Step 8, Loss: 5.0960
Epoch 1, Step 12, Loss: 4.8803
Epoch 1, Step 16, Loss: 5.7639
Epoch 1, Step 20, Loss: 5.0247
Epoch 1, Step 24, Loss: 4.2351
Epoch 1, Step 28, Loss: 4.4391
Epoch 1, Step 32, Loss: 4.9045
Epoch 1, Step 36, Loss: 4.4421
Epoch 1, Step 40, Loss: 3.7215
Epoch 1, Step 44, Loss: 4.2234
Epoch 1, Step 48, Loss: 2.6679
Epoch 1, Step 52, Loss: 3.3786
Epoch 1, Step 56, Loss: 1.8866
Epoch 1, Step 60, Loss: 2.1394
Epoch 1, Step 64, Loss: 2.0607
Epoch 1, Step 68, Loss: 2.6768
Epoch 1, Step 72, Loss: 1.1936
Epoch 1, Step 76, Loss: 1.5949
Epoch 1, Step 80, Loss: 0.8128
Epoch 1, Step 84, Loss: 0.8590
Epoch 1, Step 88, Loss: 1.9256
Epoch 1, Step 92, Loss: 1.1399
Epoch 1, Step 96, Loss: 0.5828
Epoch 1, Step 100, Loss: 1.3943
Epoch 1, Step 104, Loss: 1.2800
Epoch 1, Step 108, Loss: 0.7577
Epoch 1, Step 112, Loss: 0.3968
Epoch 1, Step 116, Loss: 0.6584
Epoch 1, Step 120, Loss: 0.7637
Epoch 1, Step 124, Loss: 0.6128
Epoch 1, Step 128, Loss: 0.2584
Ep

## Inference After Fine-Tuning

In [46]:
save_dir = "gemma-lora-adapter"

In [47]:
peft_model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

('gemma-lora-adapter/tokenizer_config.json',
 'gemma-lora-adapter/special_tokens_map.json',
 'gemma-lora-adapter/chat_template.jinja',
 'gemma-lora-adapter/tokenizer.model',
 'gemma-lora-adapter/added_tokens.json',
 'gemma-lora-adapter/tokenizer.json')

In [48]:
device

'cuda'

In [49]:
base_model_id = "google/gemma-3-270m-it" 
# adapter_path = "gemma-lora-adapter"

tokenizer = AutoTokenizer.from_pretrained(save_dir)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float32,
    device_map="auto",
    attn_implementation="eager",
)

model = PeftModel.from_pretrained(base_model, save_dir)

`torch_dtype` is deprecated! Use `dtype` instead!


In [50]:
print("Tokenizer vocab size:", len(tokenizer))
print("Model embedding size:", model.get_input_embeddings().weight.shape[0])

Tokenizer vocab size: 262145
Model embedding size: 262144


In [51]:
if device == "mps":
    model.to("cpu")
model.resize_token_embeddings(len(tokenizer))
model.to(device)
model.eval();

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [52]:
print("Tokenizer vocab size:", len(tokenizer))
print("Model embedding size:", model.get_input_embeddings().weight.shape[0])

Tokenizer vocab size: 262145
Model embedding size: 262145


In [53]:
for name, param in model.named_parameters():
    print(name, param.device)
    break

base_model.model.model.embed_tokens.weight cuda:0


In [54]:
messages = [
    {"role": "user", "content": "You are a assistant responsible for classifying mental health status. I am bored and sad"}
]

input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True,).to(device)
attention_mask = torch.ones_like(input_ids).to(device)

In [55]:
with torch.no_grad():
    outputs = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=100,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        temperature=0.1,
    )

input_len = input_ids.shape[1]
generated_tokens_tensor = outputs[0, input_len:]
decoded_response = tokenizer.decode(generated_tokens_tensor, skip_special_tokens=True)

print(decoded_response)

Based on what you've described, this sounds like 'depression'.


In [56]:
model.eval()
predictions = []
references = []
samples = 10

for batch in val_dataloader:
    input_ids = batch["prompt_input_ids"].to(device)
    attention_mask = batch["prompt_attention_mask"].to(device)
    labels = batch["labels"].to(device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=20,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
             temperature=0.1,
        )


    pred = outputs[0, input_ids.shape[1]:]

    decoded_pred = tokenizer.batch_decode(
        pred,
        skip_special_tokens=True,
    )

    labels_decoded = tokenizer.batch_decode(
        labels[0][labels[0] != -100],
        skip_special_tokens=True,
    )

    predictions.append("".join(decoded_pred))
    references.append("".join(labels_decoded))

    
    if samples <= 0:
        break
    samples -= 1
    print();

In [57]:
predictions

["Based on what you've described, this sounds like 'anxiety'.",
 "Based on what you've described, this sounds like 'anxiety'.",
 "Based on what you've described, this sounds like 'intimacy'.",
 "Based on what you've described, this sounds like 'intimacy'.",
 "Based on what you've described, this sounds like 'anxiety'.",
 "Based on what you've described, this sounds like 'behavioral-change'.",
 "Based on what you've described, this sounds like 'relationships'.",
 "Based on what you've described, this sounds like 'parenting'.",
 "Based on what you've described, this sounds like 'counseling-fundamentals'.",
 "Based on what you've described, this sounds like 'intimacy'.",
 "Based on what you've described, this sounds like 'anxiety'."]

In [58]:
references

["Based on what you've described, this sounds like 'anxiety'.\n",
 "Based on what you've described, this sounds like 'anxiety'.\n",
 "Based on what you've described, this sounds like 'marriage'.\n",
 "Based on what you've described, this sounds like 'intimacy'.\n",
 "Based on what you've described, this sounds like 'anxiety'.\n",
 "Based on what you've described, this sounds like 'behavioral-change'.\n",
 "Based on what you've described, this sounds like 'relationships'.\n",
 "Based on what you've described, this sounds like 'parenting'.\n",
 "Based on what you've described, this sounds like 'counseling-fundamentals'.\n",
 "Based on what you've described, this sounds like 'intimacy'.\n",
 "Based on what you've described, this sounds like 'anxiety'.\n"]

In [59]:
input_ids.shape, attention_mask.shape, labels.shape, outputs.shape

(torch.Size([1, 83]),
 torch.Size([1, 83]),
 torch.Size([1, 99]),
 torch.Size([1, 99]))

## Merging the Model

In [60]:
merged_model = model.merge_and_unload()

merged_model.save_pretrained("./gemma-merged")
tokenizer.save_pretrained("./gemma-merged");

In [63]:
tuned_model = AutoModelForCausalLM.from_pretrained("./gemma-merged", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("./gemma-merged")

# chatbot = pipeline("text-generation", model=tuned_model, tokenizer=tokenizer, device=0);

The tokenizer you are loading from './gemma-merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [66]:
messages = [
    {"role": "user", "content": "You are a assistant responsible for classifying mental health status. I feel anxious and stressed all the time."}
]


input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(device)
attention_mask = torch.ones_like(input_ids).to(device)

outputs = tuned_model.generate(
    input_ids,
    attention_mask=attention_mask,
    max_new_tokens=100,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
    temperature=0.1,
)

input_len = input_ids.shape[1]
generated_tokens_tensor = outputs[0, input_len:]
decoded_response = tokenizer.decode(generated_tokens_tensor, skip_special_tokens=True)

print(decoded_response)

Based on what you've described, this sounds like 'stress'.


In [ ]:
# training_args = TrainingArguments(
#     output_dir="./gemma-finetuned-model",
#     per_device_train_batch_size=4,
#     num_train_epochs=3,
#     logging_dir='./logs',
#     # save_steps=500,
#     logging_steps=100,
#     save_strategy="no",
#     label_names=["labels"],  # Explicitly specify label names for PEFT models
#     save_total_limit=1,
#     report_to="none",
#     learning_rate=0.0001,
# )

# data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# trainer = Trainer(
#     model=peft_model,
#     args=training_args,
#     train_dataset=tokenized_dataset["train"],
#     processing_class=tokenizer,
#     data_collator=data_collator,
# );

# trainer.train()
# trainer.save_model("./finetuned-model-gemma")

In [67]:
!ls

gemma-lora-adapter  gemma-merged  sample_data


In [ ]:
!zip -r /content/file.zip /content/gemma-lora-adapter
from google.colab import files
files.download("/content/file.zip")

  adding: content/gemma-lora-adapter/ (stored 0%)
  adding: content/gemma-lora-adapter/adapter_model.safetensors (deflated 7%)
  adding: content/gemma-lora-adapter/special_tokens_map.json (deflated 77%)
  adding: content/gemma-lora-adapter/added_tokens.json (stored 0%)
  adding: content/gemma-lora-adapter/adapter_config.json (deflated 58%)
  adding: content/gemma-lora-adapter/tokenizer_config.json (deflated 97%)
  adding: content/gemma-lora-adapter/chat_template.jinja (deflated 70%)
  adding: content/gemma-lora-adapter/README.md (deflated 65%)
  adding: content/gemma-lora-adapter/tokenizer.model (deflated 52%)
  adding: content/gemma-lora-adapter/tokenizer.json (deflated 83%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>